In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np

import altair as alt

from scipy.interpolate import BSpline

from mixres.sim import GaussianDisjoint1D

import altair as alt

In [26]:
# Generate the dataset
df = GaussianDisjoint1D(interval_width=20, sigma2=0.1, seed=42).generate()

# Visualize the dataset
base = alt.Chart(df).mark_point().encode(x="x", y="y")
true_line = alt.Chart(df).mark_line(color="red").encode(x="x", y="f")
mean_line = alt.Chart(df).mark_line(color="blue").encode(x="x", y="mu")
base + true_line + mean_line

alt.LayerChart(...)

In [27]:
def make_bspline_basis(
    x: np.ndarray,
    n_basis: int = 30,
    knots: np.ndarray | None = None,
) -> np.ndarray:
    """Construct a cubic B-spline basis matrix for input grid x.

    Parameters
    ----------
    x : np.ndarray, shape (N,)
        Input grid values (1-D).
    n_basis : int, optional
        Number of basis functions (default 30). Ignored when `knots` is given.
    knots : np.ndarray or None, optional
        Interior knot locations. If None, knots are placed uniformly in
        (x.min(), x.max()) such that the basis has `n_basis` functions.
        When supplied, `n_basis` is inferred as len(knots) + degree + 1.

    Returns
    -------
    np.ndarray, shape (N, M)
        Dense basis matrix, where M = n_basis (or len(knots) + degree + 1).
    """
    degree = 3
    x = np.asarray(x, dtype=float)
    x_min, x_max = x.min(), x.max()

    if knots is None:
        n_interior = n_basis - degree - 1
        if n_interior < 0:
            raise ValueError(
                f"n_basis={n_basis} is too small for degree={degree}; "
                f"need n_basis >= {degree + 1}."
            )
        interior_knots = np.linspace(x_min, x_max, n_interior + 2)[1:-1]
    else:
        interior_knots = np.asarray(knots, dtype=float)
        n_basis = len(interior_knots) + degree + 1  # override

    # Clamped knot vector: boundary knots repeated (degree + 1) times
    t = np.concatenate(
        [
            np.repeat(x_min, degree + 1),
            interior_knots,
            np.repeat(x_max, degree + 1),
        ]
    )

    B = BSpline.design_matrix(x, t, degree).toarray()  # shape (N, n_basis)
    return B

In [53]:
x = df["x"].values

# Default: 40 basis functions, uniform knot placement
B = make_bspline_basis(x, n_basis=40)
print(f"Shape          : {B.shape}")  # (N, 40)
print(f"Partition of unity: {np.allclose(B.sum(axis=1), 1.0)}")  # True
print(f"Non-negative   : {(B >= 0).all()}")  # True

# Custom knots: 10 interior knots -> 14 basis functions
B_custom = make_bspline_basis(x, knots=np.linspace(x.min(), x.max(), 12)[1:-1])
print(f"\nCustom knots shape: {B_custom.shape}")  # (N, 14)

Shape          : (101, 40)
Partition of unity: True
Non-negative   : True

Custom knots shape: (101, 14)


In [54]:
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import autoguide

from mixres.models._inference import (
    run_inference_mcmc,
    run_inference_svi,
    posterior_predictive_mcmc,
    posterior_predictive_svi,
)

In [55]:
def model(B, y_obs=None):
    """
    Bayesian B-spline regression model.

    Priors:
        beta0  ~ N(0, 1)            global intercept
        sigma2 ~ Gamma(2, 1)        observation noise variance
        w      ~ N(0, I_K)          B-spline coefficients

    Likelihood:
        y | beta0, w, sigma2 ~ N(beta0 + B @ w, sigma2)
    """
    K = B.shape[1]

    # --- Priors ---
    beta0 = numpyro.sample("beta0", dist.Normal(0, 1))
    sigma2 = numpyro.sample("sigma2", dist.Gamma(2, 1))
    w = numpyro.sample("w", dist.Normal(0, 1).expand([K]).to_event(1))

    # --- Latent mean ---
    mu = numpyro.deterministic("mu", beta0 + B @ w)

    # --- Likelihood ---
    with numpyro.plate("obs", B.shape[0]):
        numpyro.sample("y", dist.Normal(mu, jnp.sqrt(sigma2)), obs=y_obs)

In [56]:
# Convert to JAX arrays
y_obs = jnp.array(df["y"].values)
B_jax = jnp.array(B)

In [57]:
# MCMC inference with NUTS
rng_key_mcmc = jax.random.PRNGKey(0)
mcmc = run_inference_mcmc(rng_key_mcmc, model, B=B_jax, y_obs=y_obs)
mcmc_samples = mcmc.get_samples()
print(mcmc_samples.keys())

/Users/shozen/Imperial/0_Research/mixres/mixres/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1000/1000 [00:00<00:00, 3342.10it/s, 31 steps of size 1.28e-01. acc. prob=0.95]


Number of divergences: 0
dict_keys(['beta0', 'mu', 'sigma2', 'w'])


In [58]:
# SVI inference with AutoNormal guide
rng_key_svi = jax.random.PRNGKey(1)
guide = autoguide.AutoNormal(model)
svi_result = run_inference_svi(rng_key_svi, model, guide, B=B_jax, y_obs=y_obs)
print(svi_result.params.keys())

100%|██████████| 5000/5000 [00:00<00:00, 6851.77it/s, init loss: 310.4945, avg. loss [4751-5000]: 70.1290]


dict_keys(['beta0_auto_loc', 'beta0_auto_scale', 'sigma2_auto_loc', 'sigma2_auto_scale', 'w_auto_loc', 'w_auto_scale'])


In [59]:
# Extract posterior mean of mu from MCMC samples
mu = mcmc_samples["mu"].mean(axis=0)
df_results = df.copy()
df_results["mu"] = mu

# Visualize results
base = alt.Chart(df_results).mark_point().encode(x="x", y="y")
true_line = alt.Chart(df_results).mark_line(color="red").encode(x="x", y="f")
mean_line = alt.Chart(df_results).mark_line(color="blue").encode(x="x", y="mu")
base + true_line + mean_line

alt.LayerChart(...)

In [60]:
from scipy.special import factorial
from jax.typing import ArrayLike


def diff_matrix(num_nodes: int, order: int) -> ArrayLike:
    """
    Construct a finite difference matrix of given order.

    Parameters
    ----------
    num_nodes: int
        The number of nodes in the grid.
    order: int
        The order of the finite difference.

    Returns
    -------
    D: ArrayLike
        The finite difference matrix of shape (num_nodes - order, num_nodes).
    """
    D = jnp.zeros((num_nodes - order, num_nodes))
    i_vals = jnp.arange(order + 1)
    coeff = (factorial(order) / (factorial(i_vals) * factorial(order - i_vals))) * (
        -1
    ) ** (order - i_vals)
    for i in range(num_nodes - order):
        D = D.at[i, i : i + order + 1].set(coeff)

    return D

In [61]:
def model_pspline(
    B: ArrayLike, D: ArrayLike, y_obs: ArrayLike = None, epsilon: float = 1e-3
):
    """
    Bayesian P-spline regression model.

    Parameters
    ----------
    B: ArrayLike
        B-spline basis matrix of shape (N, K).
    D: ArrayLike
        Finite difference matrix of shape (K - order, K) for the penalty.
    y_obs: ArrayLike, optional
        Observed data of shape (N,). If None, the model is treated as a generative model.
    epsilon: float, optional
                Soft constraint strength (default 1e-3). Smaller values enforce smoother solutions.

    Priors:
        beta0  ~ N(0, 1)            global intercept
        sigma2 ~ Gamma(2, 1)        observation noise variance
        w      ~ N(0, I_K)          B-spline coefficients

    Likelihood:
        y | beta0, w, sigma2 ~ N(beta0 + B @ w, sigma2)
    """
    K = B.shape[1]

    # --- Priors ---
    beta0 = numpyro.sample("beta0", dist.Normal(0, 1))
    sigma2 = numpyro.sample("sigma2", dist.Gamma(2, 1))

    # Prior on the differences
    delta = numpyro.sample("delta", dist.Normal(0, 1).expand([D.shape[0]]).to_event(1))
    w = numpyro.sample("w", dist.Normal(0, 1).expand([K]).to_event(1))

    # 4. The Soft Constraints (Vectorized)
    diff = jnp.matmul(D, w)
    penalty = -0.5 * jnp.sum(jnp.square(diff - delta)) / (epsilon**2)
    numpyro.factor("soft_constraint", penalty)

    # --- Latent mean ---
    mu = numpyro.deterministic("mu", beta0 + B @ w)

    # --- Likelihood ---
    with numpyro.plate("obs", B.shape[0]):
        numpyro.sample("y", dist.Normal(mu, jnp.sqrt(sigma2)), obs=y_obs)

In [62]:
# Convert to JAX arrays
y_obs = jnp.array(df["y"].values)
B_jax = jnp.array(B)
D_jax = jnp.array(diff_matrix(B.shape[1], order=2))

In [63]:
# MCMC inference with NUTS
rng_key_mcmc = jax.random.PRNGKey(0)
mcmc = run_inference_mcmc(rng_key_mcmc, model_pspline, B=B_jax, D=D_jax, y_obs=y_obs)
mcmc_samples = mcmc.get_samples()
print(mcmc_samples.keys())

/Users/shozen/Imperial/0_Research/mixres/mixres/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1000/1000 [00:04<00:00, 207.77it/s, 1023 steps of size 8.94e-04. acc. prob=0.83]

Number of divergences: 0
dict_keys(['beta0', 'delta', 'mu', 'sigma2', 'w'])


In [64]:
# Extract posterior mean of mu from MCMC samples
mu = mcmc_samples["mu"].mean(axis=0)
lower = jnp.percentile(mcmc_samples["mu"], 2.5, axis=0)
upper = jnp.percentile(mcmc_samples["mu"], 97.5, axis=0)
df_results = df.copy()
df_results["mu"] = mu
df_results["lower"] = lower
df_results["upper"] = upper

# Visualize results
base = alt.Chart(df_results).mark_point().encode(x="x", y="y")
true_line = alt.Chart(df_results).mark_line(color="red").encode(x="x", y="f")
mean_line = alt.Chart(df_results).mark_line(color="blue").encode(x="x", y="mu")
bands = (
    alt.Chart(df_results)
    .mark_errorband(extent="ci")
    .encode(x="x", y="lower", y2="upper")
)
base + true_line + mean_line + bands

alt.LayerChart(...)

In [65]:
def model_fHS(
    B: ArrayLike, D: ArrayLike, y_obs: ArrayLike = None, epsilon: float = 1e-3
):
    """
    Bayesian P-spline regression model.

    Parameters
    ----------
    B: ArrayLike
        B-spline basis matrix of shape (N, K).
    D: ArrayLike
        Finite difference matrix of shape (K - order, K) for the penalty.
    y_obs: ArrayLike, optional
        Observed data of shape (N,). If None, the model is treated as a generative model.
    epsilon: float, optional
                Soft constraint strength (default 1e-3). Smaller values enforce smoother solutions.

    Priors:
        beta0  ~ N(0, 1)            global intercept
        sigma2 ~ Gamma(2, 1)        observation noise variance
        w      ~ N(0, I_K)          B-spline coefficients

    Likelihood:
        y | beta0, w, sigma2 ~ N(beta0 + B @ w, sigma2)
    """
    K = B.shape[1]

    # --- Priors ---
    beta0 = numpyro.sample("beta0", dist.Normal(0, 1))
    sigma2 = numpyro.sample("sigma2", dist.Gamma(2, 1))

    # Prior on the differences
    tau = numpyro.sample("tau", dist.HalfCauchy(1.0))
    lam = numpyro.sample("_lambda", dist.HalfCauchy(1.0).expand([D.shape[0]]))
    delta = numpyro.sample("delta", dist.Normal(0, 1).expand([D.shape[0]]).to_event(1))
    delta = delta * tau**2 * lam**2

    w = numpyro.sample("w", dist.Normal(0, 1).expand([K]).to_event(1))

    # 4. The Soft Constraints (Vectorized)
    diff = jnp.matmul(D, w)
    penalty = -0.5 * jnp.sum(jnp.square(diff - delta)) / (epsilon**2)
    numpyro.factor("soft_constraint", penalty)

    # --- Latent mean ---
    mu = numpyro.deterministic("mu", beta0 + B @ w)

    # --- Likelihood ---
    with numpyro.plate("obs", B.shape[0]):
        numpyro.sample("y", dist.Normal(mu, jnp.sqrt(sigma2)), obs=y_obs)

In [66]:
# MCMC inference with NUTS
rng_key_mcmc = jax.random.PRNGKey(0)
mcmc = run_inference_mcmc(rng_key_mcmc, model_fHS, B=B_jax, D=D_jax, y_obs=y_obs)
mcmc_samples = mcmc.get_samples()
print(mcmc_samples.keys())

/Users/shozen/Imperial/0_Research/mixres/mixres/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1000/1000 [00:06<00:00, 146.66it/s, 1023 steps of size 5.64e-04. acc. prob=0.92]

Number of divergences: 70
dict_keys(['_lambda', 'beta0', 'delta', 'mu', 'sigma2', 'tau', 'w'])


In [68]:
# Extract posterior mean of mu from MCMC samples
mu = mcmc_samples["mu"].mean(axis=0)
lower = jnp.percentile(mcmc_samples["mu"], 2.5, axis=0)
upper = jnp.percentile(mcmc_samples["mu"], 97.5, axis=0)
df_results = df.copy()
df_results["mu"] = mu
df_results["lower"] = lower
df_results["upper"] = upper

# Visualize results
base = alt.Chart(df_results).mark_point().encode(x="x", y="y")
true_line = alt.Chart(df_results).mark_line(color="red").encode(x="x", y="f")
mean_line = alt.Chart(df_results).mark_line(color="blue").encode(x="x", y="mu")
bands = (
    alt.Chart(df_results)
    .mark_errorband(extent="ci")
    .encode(x="x", y="lower", y2="upper")
)
base + true_line + mean_line + bands

alt.LayerChart(...)